## TRABAJO GRUPAL SISTEMA RECOMENDACIONES

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import random

# Inicializar sesión de Spark
spark = SparkSession.builder.appName("RecomendadorALS").getOrCreate()

# 1. Cargar tu CSV (cambia la ruta por la tuya)
# El CSV debe contener al menos: categoria, marca, tienda, precio
df_productos_raw = spark.read.csv("productos.csv", header=True, inferSchema=True)

# 2. Asegurar un ID numérico secuencial para los productos
windowSpec = Window.orderBy("categoria", "marca")
df_productos = df_productos_raw.withColumn("product_id", F.row_number().over(windowSpec))
df_productos_raw.printSchema()
print(f"Productos cargados: {df_productos.count()}")
df_productos.show(5)

root
 |-- id_producto: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- marca: string (nullable = true)
 |-- tienda: string (nullable = true)
 |-- precio: double (nullable = true)

Productos cargados: 35
+-----------+--------------------+-----------------+--------------+-------------------+------+----------+
|id_producto|              nombre|        categoria|         marca|             tienda|precio|product_id|
+-----------+--------------------+-----------------+--------------+-------------------+------+----------+
|         29|Aire Acondicionad...|    Climatizacion|            LG|Creditos Economicos|469.99|         1|
|         30|Ventilador de Ped...|    Climatizacion|        Taurus|Creditos Economicos|  45.0|         2|
|         27|Freidora de Aire ...|Electrodomesticos|Black & Decker|Creditos Economicos|  85.0|         3|
|         28|Cafetera Hamilton...|Electrodomesticos|Hamilton Beach|Creditos Economicos|  55.0|     

## PERFILES DINAMICACOS Y SIMULACION DE CLIENTES

In [4]:
import random
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType

# Cargar el dataframe de productos
df_productos = spark.read.csv("productos.csv", header=True, inferSchema=True)

# Extraer listas de categorías y marcas reales de tu CSV para que el script las conozca
todas_categorias = [row['categoria'] for row in df_productos.select('categoria').distinct().collect() if row['categoria']]
todas_marcas = [row['marca'] for row in df_productos.select('marca').distinct().collect() if row['marca']]

print("Categorías detectadas en tu CSV:", todas_categorias)
print("Marcas detectadas en tu CSV:", todas_marcas)

# Mapeo inteligente: Intentamos asociar tus datos reales a los perfiles requeridos
def buscar_coincidencias(lista_origen, palabras_clave):
    return [item for item in lista_origen if any(pc.lower() in item.lower() for pc in palabras_clave)]

# Construcción dinámica de perfiles basados en TUS datos
perfiles_config = {
    "tecnologico": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["tecno", "electro", "compu", "celular", "audio"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["apple", "samsung", "asus", "sony", "hp", "lg", "xiaomi"]),
        "presupuesto_max": 1500.0
    },
    "gamer": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["videojuego", "gamer", "juego", "consola", "compu"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["sony", "nintendo", "asus", "razer", "logitech", "playstation"]),
        "presupuesto_max": 1200.0
    },
    "hogar": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["hogar", "mueble", "linea blanca", "cocina", "electrodomestico"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["indurama", "lg", "whirlpool", "mabe", "samsung"]),
        "presupuesto_max": 800.0
    },
    "estudiante": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["libro", "papeleria", "utiles", "cuaderno", "compu"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["hp", "lenovo", "dell", "epson", "bic", "artesco"]),
        "presupuesto_max": 350.0
    },
    "fitness": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["deporte", "fitness", "suplemento", "ropa", "ejercicio"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["nike", "adidas", "puma", "under armour", "reebok"]),
        "presupuesto_max": 250.0
    },
    "premium": {
        "categorias_afines": todas_categorias, # Le interesa todo lo caro de cualquier categoría
        "marcas_afines": buscar_coincidencias(todas_marcas, ["apple", "sony", "samsung", "bose", "nike"]),
        "presupuesto_max": 99999.0 # Sin límite práctico
    },
    "ahorrador": {
        "categorias_afines": buscar_coincidencias(todas_categorias, ["supermercado", "despensa", "comida", "limpieza"]),
        "marcas_afines": buscar_coincidencias(todas_marcas, ["generica", "maggi", "la fabril", "gullon", "snob"]),
        "presupuesto_max": 40.0
    }
}

Categorías detectadas en tu CSV: ['TV y Audio', 'Climatizacion', 'Electrodomesticos', 'Tecnologia', 'Hogar']
Marcas detectadas en tu CSV: ['Sony', 'Huawei', 'Xiaomi', 'Taurus', 'HP', 'Mabe', 'Hamilton Beach', 'Asus', 'Lenovo', 'Samsung', 'Black & Decker', 'Nintendo', 'Generico', 'Indurama', 'HyperX', 'LG', 'Honor', 'Apple', 'Chaide', 'Oster', 'Xtratech', 'TCL']


In [5]:
# 1. Generar 30 clientes con perfiles aleatorios
lista_perfiles = list(perfiles_config.keys())
datos_clientes = [(user_id, random.choice(lista_perfiles)) for user_id in range(1, 31)]
df_clientes = spark.createDataFrame(datos_clientes, ["user_id", "perfil"])

# 2. Enviar la configuración a los nodos de Spark
perfiles_bc = spark.sparkContext.broadcast(perfiles_config)

# 3. Función para calcular el Rating sugerido por la guía
def calcular_rating_logica(perfil, categoria, marca, precio):
    config = perfiles_bc.value.get(perfil)
    if not config: return 2.5

    base = 2.5
    afinidad_cat = 1.2 if categoria in config["categorias_afines"] else 0.0
    afinidad_marca = 0.6 if marca in config["marcas_afines"] else 0.0

    # Penalización por precio
    if precio > config["presupuesto_max"]:
        penalizacion_precio = 1.8  # Se sale de su presupuesto
    else:
        penalizacion_precio = (precio / config["presupuesto_max"]) * 0.4

    # Ruido controlado
    ruido = random.uniform(-0.3, 0.3)

    # Total y límites (1 a 5)
    rating_final = base + afinidad_cat + afinidad_marca - penalizacion_precio + ruido
    return float(max(1.0, min(5.0, rating_final)))

rating_udf = F.udf(calcular_rating_logica, FloatType())

# 4. Generar interacciones cruzando clientes y productos (Producto Cartesiano)
df_matriz_completa = df_clientes.crossJoin(df_productos)

# 5. Calcular los ratings calculados
df_matriz_utilidad = df_matriz_completa.withColumn(
    "rating",
    rating_udf(F.col("perfil"), F.col("categoria"), F.col("marca"), F.col("precio"))
)

# 6. Tomar una muestra aleatoria para simular escasez (Ej: conservar ~300 interacciones de las >1000 posibles)
# Esto garantiza cumplir con el mínimo de 250 interacciones requerido
df_interacciones_final = df_matriz_utilidad.sample(withReplacement=False, fraction=0.35, seed=42)

print(f"Total de interacciones simuladas logradas: {df_interacciones_final.count()}")
df_interacciones_final.select("user_id", "perfil", "nombre", "categoria", "rating").show(10, truncate=False)

Total de interacciones simuladas logradas: 372
+-------+------+----------------------------------------+-----------------+---------+
|user_id|perfil|nombre                                  |categoria        |rating   |
+-------+------+----------------------------------------+-----------------+---------+
|1      |gamer |Celular Honor X8a 128GB                 |Tecnologia       |2.5853038|
|1      |gamer |Laptop Lenovo IdeaPad 3 Ryzen 5         |Tecnologia       |2.0628605|
|1      |gamer |Consola Nintendo Switch OLED 64GB       |TV y Audio       |2.8347754|
|1      |gamer |Reloj Inteligente Huawei Watch GT 4     |Tecnologia       |2.6363053|
|1      |gamer |Licuadora Oster Reversible 2 Velocidades|Electrodomesticos|2.3228543|
|1      |gamer |Freidora de Aire Black & Decker 4.5L    |Electrodomesticos|2.2032502|
|1      |gamer |Cafetera Hamilton Beach de Goteo 12tz   |Electrodomesticos|2.599844 |
|1      |gamer |Sofa Cama Plegable Microfibra           |Hogar            |2.2451355|
|2     

## ENTRENAMIENTO Y VALIDACION  DE MODELO ALS

In [6]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Dividir en Entrenamiento (80%) y Prueba (20%)
(train, test) = df_interacciones_final.randomSplit([0.8, 0.2], seed=42)

# Configurar ALS (usando las columnas exactas de tu archivo)
als = ALS(
    userCol="user_id",
    itemCol="id_producto", # Tu columna del CSV
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

# Entrenar el modelo con hiperparámetros estándar ajustables
model = als.setRank(10).setRegParam(0.1).fit(train)

# Predicciones de prueba
predictions = model.transform(test)

# Evaluadores de error (RMSE y MAE)
evaluator_rmse = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator_rmse.evaluate(predictions)

evaluator_mae = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")
mae = evaluator_mae.evaluate(predictions)

print(f">>> Métrica RMSE: {rmse:.4f}")
print(f">>> Métrica MAE: {mae:.4f}")

>>> Métrica RMSE: 0.4364
>>> Métrica MAE: 0.3281


## RECOMENDACIONES PARA 5 CLIENTES

In [7]:
# Obtener las 5 mejores recomendaciones para cada usuario
userRecs = model.recommendForAllUsers(5)

# Tomar una muestra de 5 usuarios para el reporte
df_recs_5 = userRecs.limit(5)

# Expandir la estructura interna de las recomendaciones de Spark
df_recs_exploded = df_recs_5.withColumn("recommendation", F.explode("recommendations")) \
    .select("user_id", F.col("recommendation.id_producto").alias("id_producto"), F.col("recommendation.rating").alias("predicted_rating"))

# Cruzar con los datos originales de tus productos para que sea legible
df_reporte_final = df_recs_exploded.join(df_productos, on="id_producto", how="inner") \
    .select("user_id", "nombre", "categoria", "marca", "tienda", "precio", "predicted_rating") \
    .orderBy("user_id", F.desc("predicted_rating"))

# Mostrar el resultado final esperado en el documento
df_reporte_final.show(25, truncate=False)

+-------+----------------------------------------+-----------------+--------------+-------------------+-------+----------------+
|user_id|nombre                                  |categoria        |marca         |tienda             |precio |predicted_rating|
+-------+----------------------------------------+-----------------+--------------+-------------------+-------+----------------+
|1      |Microondas Mabe 20 Litros Negro         |Electrodomesticos|Mabe          |Creditos Economicos|109.99 |2.6842356       |
|1      |Escritorio Gamer Xtratech Fibra Carbono |Hogar            |Xtratech      |Creditos Economicos|119.99 |2.5515263       |
|1      |Cafetera Hamilton Beach de Goteo 12tz   |Electrodomesticos|Hamilton Beach|Creditos Economicos|55.0   |2.5298796       |
|1      |Celular Xiaomi Redmi Note 12 128GB      |Tecnologia       |Xiaomi        |Creditos Economicos|199.99 |2.4923363       |
|1      |Freidora de Aire Black & Decker 4.5L    |Electrodomesticos|Black & Decker|Creditos Econo